# survivor 탐지 모델 비교 (Colab)

고정 웹캠 출구 인형 카운트에 쓸 탐지 모델을 **같은 데이터·같은 조건**으로 학습해 비교한다.

| 계열 | 모델 |
| --- | --- |
| YOLO (Ultralytics) | yolov8n · yolov8s · yolo11n · yolo11s · yolo26n · yolo26s |
| DETR (Ultralytics) | rtdetr-l |

**실행 방법**
1. 메뉴 `런타임 → 런타임 유형 변경 → GPU(T4 이상)`.
2. Google Drive 의 `survivor_model_compare/` 폴더에 통합 데이터셋 zip(`survivor-v2-merged.zip`: train = v1 학습 영상, valid = VAL 영상, test = TEST 영상)을 올린다.
3. `런타임 → 모두 실행`. 끊겨도 다시 `모두 실행`하면 **끝난 모델은 건너뛰고** 이어서 한다(결과는 Drive 에 저장).

**결과**: Drive `survivor_model_compare/results/` — `notion_summary.md`(Notion 붙여넣기용), `results.csv`(결과표), `hyperparameters.csv`, `augmentation.csv`, `dataset_info.json`, `summary.json`(전부), 그래프 PNG, 모델별 가중치·학습 로그·그래프.

> ⚠️ **지표 해석 주의**: v2 는 valid·test 를 학습 영상과 따로 찍은 영상에서만 만들었다(v1 의 이웃 프레임 평가셋은 제외). 다만 valid 와 test 는 같은 시간대·같은 사람이 연달아 찍어 조건이 비슷하다. 두 모델 차이는 8-2 칸의 신뢰구간으로 판단하고, 구간이 0 을 포함하면 **구분 불가**로 쓴다.

## 1. GPU 확인

In [ ]:
!nvidia-smi -L
import torch
assert torch.cuda.is_available(), "GPU 런타임이 아니다: 런타임 → 런타임 유형 변경 → GPU"
print(torch.__version__, torch.cuda.get_device_name(0))

## 2. 설치
로컬 실험과 같은 Ultralytics 버전으로 고정한다(결과 재현용).

In [ ]:
!pip -q install ultralytics==8.4.138
import ultralytics, torchvision
print("ultralytics", ultralytics.__version__, "| torchvision", torchvision.__version__)

## 3. 설정
모든 모델에 같은 값을 쓴다. 바꾸지 않은 값은 Ultralytics 기본값이며 모델별 `args.yaml` 에 전부 남는다.

In [ ]:
from pathlib import Path

DRIVE_DIR = Path("/content/drive/MyDrive/survivor_model_compare")
ZIP_NAME = "survivor-v2-merged.zip"   # 통합본(v1 학습 + VAL·TEST 영상 평가)
RESULTS = DRIVE_DIR / "results"            # 끊겨도 남도록 Drive 에 저장
DATA = Path("/content/data/survivor-v1")   # 압축은 Colab 로컬 디스크에 푼다(빠름)
RUNS = Path("/content/runs")

EPOCHS = 100
IMGSZ = 640
BATCH = 16
SEED = 0

ULTRALYTICS_MODELS = ["yolov8n", "yolo11n", "yolo26n", "yolov8s", "yolo11s", "yolo26s", "rtdetr-l"]
BATCH_OVERRIDE = {"rtdetr-l": 8}           # RT-DETR-l 은 메모리가 커서 8
# 2026-10-11 고정 카메라 담당 결정: 데이터가 늘어 시간이 길어지므로 YOLO 6개 + RT-DETR-l 만 비교한다.
# torchvision(Faster R-CNN·SSDLite)을 다시 넣으려면 ["frcnn_mnv3", "ssdlite"] 로 바꾼다.
TORCHVISION_MODELS = []

# PATIENCE: val 지표가 이 에폭 수만큼 나아지지 않으면 멈춘다(2026-10-11 고정 카메라 담당 결정: 100 → 30).
# 주의: 평가셋을 별도 촬영분으로 확충하기 전에는 val 이 45장(인스턴스 43)뿐이고 train 과 이웃 프레임이라,
# mAP50-95 의 에폭 간 흔들림(잡음)에 patience 가 반응해 멈추는 시점이 우연에 좌우될 수 있다.
PATIENCE = 30
HP = dict(epochs=EPOCHS, imgsz=IMGSZ, batch=BATCH, seed=SEED, deterministic=True, device=0, workers=2,
          optimizer="auto", patience=PATIENCE, cos_lr=False, close_mosaic=10, plots=True, verbose=False)
TV_HP = {
    "frcnn_mnv3": dict(epochs=EPOCHS, batch=BATCH, lr=0.01, momentum=0.9, weight_decay=1e-4, warmup_epochs=1,
                       imgsz=640, hflip=0.5, seed=SEED, schedule="cosine"),
    "ssdlite": dict(epochs=EPOCHS, batch=BATCH, lr=0.01, momentum=0.9, weight_decay=4e-5, warmup_epochs=1,
                    imgsz=320, hflip=0.5, seed=SEED, schedule="cosine"),
}

## 4. Drive 연결과 데이터셋

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
import json, zipfile, glob

zip_path = DRIVE_DIR / ZIP_NAME
assert zip_path.exists(), f"Drive 에 {zip_path} 가 없다. survivor_model_compare 폴더에 zip 을 올린다"
RESULTS.mkdir(parents=True, exist_ok=True)
if not (DATA / "data.yaml").exists():
    DATA.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zip_path) as z:
        for m in z.namelist():                      # 압축 밖으로 나가는 경로는 거절
            assert not m.startswith("/") and ".." not in Path(m).parts, m
        z.extractall(DATA)

DATA_YAML = Path("/content/data/survivor.yaml")
DATA_YAML.write_text(f"path: {DATA}\ntrain: train/images\nval: valid/images\ntest: test/images\n"
                     "nc: 1\nnames: ['survivor']\n")

import hashlib
def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()

info = {"zip": ZIP_NAME, "zip_sha256": sha256(zip_path), "splits": {}}
for s in ("train", "valid", "test"):
    labels = sorted((DATA / s / "labels").glob("*.txt"))
    rows = [r.split() for f in labels for r in f.read_text().splitlines() if r.strip()]
    widths = sorted(float(r[3]) * 640 for r in rows)
    info["splits"][s] = {"images": len(labels), "boxes": len(rows),
                         "empty_images": sum(1 for f in labels if not f.read_text().strip()),
                         "box_width_px_min_median_max": [round(widths[0]), round(widths[len(widths) // 2]),
                                                         round(widths[-1])] if widths else None}
readme = (DATA / "README.roboflow.txt").read_text()
info["roboflow_preprocessing_and_augmentation"] = [l.strip("* ").strip() for l in readme.splitlines()
                                                   if l.startswith("* ") and "roboflow" not in l.lower()]
(RESULTS / "dataset_info.json").write_text(json.dumps(info, ensure_ascii=False, indent=2))
print(json.dumps(info, ensure_ascii=False, indent=2))

## 5. 공통 도구
모든 모델을 같은 방식으로 평가한다: IoU 0.5:0.95, conf 0.001, 최대 300개, Ultralytics `ap_per_class`.

In [ ]:
import time, math, random, shutil
import numpy as np
import pandas as pd

SUMMARY = RESULTS / "summary.json"
summary = json.loads(SUMMARY.read_text()) if SUMMARY.exists() else {}

FAILURES = RESULTS / "failures.json"
failures = json.loads(FAILURES.read_text()) if FAILURES.exists() else {}

def save_summary():
    SUMMARY.write_text(json.dumps(summary, ensure_ascii=False, indent=2))
    FAILURES.write_text(json.dumps(failures, ensure_ascii=False, indent=2))

def ul_metrics(res):
    d = res.results_dict
    return {"precision": round(d["metrics/precision(B)"], 4), "recall": round(d["metrics/recall(B)"], 4),
            "mAP50": round(d["metrics/mAP50(B)"], 4), "mAP50_95": round(d["metrics/mAP50-95(B)"], 4)}

VAL_IMAGES = sorted((DATA / "valid" / "images").glob("*"))

# Ultralytics 가 학습 중(온라인)에 거는 증강 값. 모델마다 실제 적용값을 trainer.args 에서 읽어 남긴다.
AUG_KEYS = ["hsv_h", "hsv_s", "hsv_v", "degrees", "translate", "scale", "shear", "perspective", "flipud", "fliplr",
            "mosaic", "mixup", "cutmix", "copy_paste", "close_mosaic", "erasing", "auto_augment"]

def jsonable(v):
    return v if isinstance(v, (int, float, str, bool, type(None))) else str(v)

## 6. Ultralytics 모델 학습 (YOLO 6개 + RT-DETR-l)
끝난 모델은 건너뛴다. 모델마다 가중치·`args.yaml`·`results.csv`·그래프를 Drive 로 복사한다.

In [ ]:
from ultralytics import YOLO, RTDETR
from ultralytics.utils.downloads import attempt_download_asset
import ultralytics

def train_ultralytics(name):
    print(f"\n===== {name} =====")
    cls = RTDETR if name.startswith("rtdetr") else YOLO
    hp = {**HP, "batch": BATCH_OVERRIDE.get(name, BATCH)}
    # 공식 사전학습 가중치(처음이면 자동 다운로드). 실제로 읽은 파일 경로로 해시를 잰다: GPU 에서는 Ultralytics 의
    # AMP 점검이 yolo26n.pt 를 설정 폴더(weights_dir)에 받아 두고, 같은 이름은 그 파일을 읽는다(2026-10-11 실패 원인).
    weights_path = attempt_download_asset(f"{name}.pt")
    model = cls(weights_path)
    pre_sha = sha256(weights_path)
    t0 = time.time()
    model.train(data=str(DATA_YAML), project=str(RUNS), name=name, exist_ok=True, **hp)
    train_min = (time.time() - t0) / 60
    run = RUNS / name
    best = run / "weights" / "best.pt"
    trained = cls(str(best))
    val = ul_metrics(trained.val(data=str(DATA_YAML), split="val", device=0, plots=False, verbose=False,
                                 project=str(RUNS / "_val"), name=name, exist_ok=True))
    test = ul_metrics(trained.val(data=str(DATA_YAML), split="test", device=0, plots=False, verbose=False,
                                  project=str(RUNS / "_test"), name=name, exist_ok=True))
    shutil.copytree(run, RESULTS / "runs" / name, dirs_exist_ok=True)
    targs = vars(model.trainer.args)
    opt = model.trainer.optimizer
    summary[name] = {"family": "Ultralytics", "framework": f"ultralytics {ultralytics.__version__}",
                     "hp": hp, "optimizer_used": type(opt).__name__,
                     "lr0_used": round(float(opt.param_groups[0].get("initial_lr", opt.param_groups[0]["lr"])), 6),
                     "momentum_used": targs.get("momentum"), "weight_decay": targs.get("weight_decay"),
                     "warmup_epochs": targs.get("warmup_epochs"), "lrf": targs.get("lrf"),
                     "online_aug": {k: jsonable(targs.get(k)) for k in AUG_KEYS if k in targs},
                     "train_args": {k: jsonable(v) for k, v in targs.items()},
                     "pretrained_sha256": pre_sha, "best_sha256": sha256(best),
                     "params_M": round(sum(p.numel() for p in trained.model.parameters()) / 1e6, 2),
                     "train_minutes": round(train_min, 1), "best_epoch": None, "val": val, "test": test,
                     "weights": str(RESULTS / "runs" / name / "weights" / "best.pt")}
    curve = pd.read_csv(run / "results.csv")
    curve.columns = [c.strip() for c in curve.columns]
    summary[name]["best_epoch"] = int(curve.loc[curve["metrics/mAP50-95(B)"].idxmax(), "epoch"])
    save_summary()
    print(f"DONE {name}: {train_min:.1f}분, val mAP50-95 {val['mAP50_95']}, test mAP50-95 {test['mAP50_95']}")

for name in ULTRALYTICS_MODELS:
    if name in summary:
        print(f"skip {name} (이미 끝남)"); continue
    try:
        train_ultralytics(name)
        failures.pop(name, None)
    except Exception as e:                 # 한 모델이 실패해도 나머지는 계속, 실패는 기록·표시한다
        import traceback; traceback.print_exc()
        failures[name] = f"{type(e).__name__}: {e}"
        print(f"FAILED {name}: {failures[name]}")
    save_summary()

## 7. torchvision 모델 (이번 비교에서는 끔)
`TORCHVISION_MODELS` 가 비어 있어 학습은 건너뛴다. 아래 칸의 평가 도구(IoU 매칭·AP 계산)는 8-2 칸이 같이 쓰므로 실행은 해야 한다.

같은 YOLO 라벨을 읽어 변환한다. 증강은 좌우 뒤집기만 쓴다(Roboflow 오프라인 증강은 train 에 이미 들어 있음). Ultralytics 의 mosaic 등 온라인 증강은 없으므로 비교할 때 이 차이를 감안한다. 매 epoch valid mAP50-95 가 가장 높은 가중치를 고른다.

In [ ]:
import torchvision
from functools import partial
from PIL import Image
from torchvision.models import detection as D
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torchvision.models.detection.ssdlite import SSDLiteClassificationHead
from torchvision.models.detection import _utils as det_utils
from torchvision.transforms.functional import to_tensor
from ultralytics.utils.metrics import ap_per_class, box_iou
import matplotlib.pyplot as plt

DEV = torch.device("cuda")
IOUV = torch.linspace(0.5, 0.95, 10)

class YoloDir(torch.utils.data.Dataset):
    def __init__(self, split, hflip=0.0):
        self.images, self.split, self.hflip = sorted((DATA / split / "images").glob("*")), split, hflip
    def __len__(self):
        return len(self.images)
    def __getitem__(self, i):
        img = Image.open(self.images[i]).convert("RGB"); w, h = img.size
        rows = [r.split() for r in (DATA / self.split / "labels" / (self.images[i].stem + ".txt")).read_text().splitlines() if r.strip()]
        boxes = torch.tensor([[(float(cx) - float(bw) / 2) * w, (float(cy) - float(bh) / 2) * h,
                               (float(cx) + float(bw) / 2) * w, (float(cy) + float(bh) / 2) * h]
                              for _, cx, cy, bw, bh in rows], dtype=torch.float32).reshape(-1, 4)
        x = to_tensor(img)
        if self.hflip and random.random() < self.hflip:
            x = x.flip(-1); boxes[:, [0, 2]] = w - boxes[:, [2, 0]]
        return x, {"boxes": boxes, "labels": torch.ones(len(boxes), dtype=torch.int64)}  # 0=배경, 1=survivor

def collate(batch):
    return tuple(zip(*batch))

def build_tv(name):
    if name == "frcnn_mnv3":
        w = D.FasterRCNN_MobileNet_V3_Large_FPN_Weights.COCO_V1
        m = D.fasterrcnn_mobilenet_v3_large_fpn(weights=w, min_size=640, max_size=640, box_score_thresh=0.001,
                                                box_detections_per_img=300)
        m.roi_heads.box_predictor = FastRCNNPredictor(m.roi_heads.box_predictor.cls_score.in_features, 2)
    else:
        w = D.SSDLite320_MobileNet_V3_Large_Weights.COCO_V1
        m = D.ssdlite320_mobilenet_v3_large(weights=w, score_thresh=0.001, detections_per_img=300)
        m.head.classification_head = SSDLiteClassificationHead(
            det_utils.retrieve_out_channels(m.backbone, (320, 320)), m.anchor_generator.num_anchors_per_location(),
            2, partial(torch.nn.BatchNorm2d, eps=0.001, momentum=0.03))
    return m, w

@torch.no_grad()
def tv_evaluate(model, split):
    """Ultralytics DetectionValidator 와 같은 매칭·AP 계산."""
    model.eval(); ds = YoloDir(split); stats = []
    for i in range(len(ds)):
        x, t = ds[i]
        out = model([x.to(DEV)])[0]
        pb, ps, gt = out["boxes"].cpu(), out["scores"].cpu(), t["boxes"]
        correct = np.zeros((len(pb), len(IOUV)), dtype=bool)
        if len(pb) and len(gt):
            iou = box_iou(gt, pb).numpy()
            for k, th in enumerate(IOUV.tolist()):
                mt = np.array(np.nonzero(iou >= th)).T
                if mt.shape[0]:
                    if mt.shape[0] > 1:
                        mt = mt[iou[mt[:, 0], mt[:, 1]].argsort()[::-1]]
                        mt = mt[np.unique(mt[:, 1], return_index=True)[1]]
                        mt = mt[np.unique(mt[:, 0], return_index=True)[1]]
                    correct[mt[:, 1].astype(int), k] = True
        stats.append((correct, ps.numpy(), np.zeros(len(pb)), np.zeros(len(gt))))
    tp, conf, pcls, tcls = (np.concatenate(s, 0) for s in zip(*stats))
    _, _, p, r, _, ap, *_ = ap_per_class(tp, conf, pcls, tcls)
    return {"precision": round(float(p[0]), 4), "recall": round(float(r[0]), 4),
            "mAP50": round(float(ap[0, 0]), 4), "mAP50_95": round(float(ap[0].mean()), 4)}

def train_torchvision(name):
    print(f"\n===== {name} =====")
    hp = TV_HP[name]
    random.seed(hp["seed"]); np.random.seed(hp["seed"]); torch.manual_seed(hp["seed"])
    model, weights_enum = build_tv(name); model.to(DEV)
    loader = torch.utils.data.DataLoader(YoloDir("train", hp["hflip"]), batch_size=hp["batch"], shuffle=True,
                                         collate_fn=collate, num_workers=2)
    opt = torch.optim.SGD([p for p in model.parameters() if p.requires_grad], lr=hp["lr"],
                          momentum=hp["momentum"], weight_decay=hp["weight_decay"])
    total, warm = hp["epochs"] * len(loader), hp["warmup_epochs"] * len(loader)
    lr_at = lambda s: hp["lr"] * (s / warm if s < warm else 0.5 * (1 + math.cos(math.pi * (s - warm) / max(1, total - warm))))
    run = RUNS / name; run.mkdir(parents=True, exist_ok=True)
    best, history, step, t0 = -1.0, [], 0, time.time()
    for ep in range(hp["epochs"]):
        model.train(); losses = []
        for xs, ts in loader:
            for g in opt.param_groups:
                g["lr"] = lr_at(step)
            loss = sum(model([x.to(DEV) for x in xs], [{k: v.to(DEV) for k, v in t.items()} for t in ts]).values())
            opt.zero_grad(); loss.backward(); opt.step(); losses.append(loss.item()); step += 1
        v = tv_evaluate(model, "valid")
        history.append({"epoch": ep + 1, "loss": round(sum(losses) / len(losses), 4), **v})
        if v["mAP50_95"] > best:
            best = v["mAP50_95"]; torch.save(model.state_dict(), run / "best.pt")
        if (ep + 1) % 10 == 0:
            print(f"{name} epoch {ep + 1}/{hp['epochs']} loss {history[-1]['loss']} val mAP50-95 {v['mAP50_95']}")
    train_min = (time.time() - t0) / 60
    hist = pd.DataFrame(history); hist.to_csv(run / "history.csv", index=False)
    fig, ax = plt.subplots(1, 2, figsize=(12, 4))
    ax[0].plot(hist["epoch"], hist["loss"]); ax[0].set_title(f"{name} train loss"); ax[0].set_xlabel("epoch")
    for k in ("mAP50", "mAP50_95", "precision", "recall"):
        ax[1].plot(hist["epoch"], hist[k], label=k)
    ax[1].set_title(f"{name} valid metrics"); ax[1].set_xlabel("epoch"); ax[1].legend(); ax[1].grid(alpha=.3)
    plt.tight_layout(); plt.savefig(run / "results.png", dpi=120); plt.show()
    model.load_state_dict(torch.load(run / "best.pt", map_location=DEV))
    val, test = tv_evaluate(model, "valid"), tv_evaluate(model, "test")
    shutil.copytree(run, RESULTS / "runs" / name, dirs_exist_ok=True)
    summary[name] = {"family": "torchvision", "framework": f"torchvision {torchvision.__version__}", "hp": hp,
                     "optimizer_used": "SGD", "lr0_used": hp["lr"], "momentum_used": hp["momentum"],
                     "weight_decay": hp["weight_decay"], "warmup_epochs": hp["warmup_epochs"], "lrf": 0.0,
                     "online_aug": {"fliplr": hp["hflip"]},
                     "pretrained": weights_enum.url, "best_sha256": sha256(run / "best.pt"),
                     "params_M": round(sum(p.numel() for p in model.parameters()) / 1e6, 2),
                     "train_minutes": round(train_min, 1),
                     "best_epoch": int(max(history, key=lambda h: h["mAP50_95"])["epoch"]),
                     "val": val, "test": test, "weights": str(RESULTS / "runs" / name / "best.pt")}
    save_summary()
    print(f"DONE {name}: {train_min:.1f}분, val mAP50-95 {val['mAP50_95']}, test mAP50-95 {test['mAP50_95']}")

for name in TORCHVISION_MODELS:
    if name in summary:
        print(f"skip {name} (이미 끝남)"); continue
    try:
        train_torchvision(name)
        failures.pop(name, None)
    except Exception as e:                 # 한 모델이 실패해도 나머지는 계속, 실패는 기록·표시한다
        import traceback; traceback.print_exc()
        failures[name] = f"{type(e).__name__}: {e}"
        print(f"FAILED {name}: {failures[name]}")
    save_summary()

## 8. 추론 속도 (같은 방법으로 측정)
배치 1, valid 이미지 한 장씩, 전처리+추론+후처리를 포함한 **끝에서 끝까지** 시간. 워밍업 10장 뒤 평균. GPU(Colab) 와 CPU 둘 다 잰다. CPU 값은 PC3(웹캠 PC) 가 GPU 없이 돌릴 때의 대략적인 참고치다(Colab CPU 성능은 PC3 와 다르다).

In [ ]:
def bench_ultralytics(weights, device, n=60):
    m = (RTDETR if "rtdetr" in str(weights) else YOLO)(weights)
    times = []
    for i, p in enumerate((VAL_IMAGES * 3)[:n]):
        if device != "cpu": torch.cuda.synchronize()
        t0 = time.perf_counter()
        m.predict(str(p), imgsz=IMGSZ, device=device, conf=0.25, verbose=False)
        if device != "cpu": torch.cuda.synchronize()
        if i >= 10: times.append((time.perf_counter() - t0) * 1000)
    return round(float(np.mean(times)), 1)

def bench_tv(name, weights, device, n=60):
    m, _ = build_tv(name); m.load_state_dict(torch.load(weights, map_location="cpu")); m.eval().to(device)
    times = []
    with torch.no_grad():
        for i, p in enumerate((VAL_IMAGES * 3)[:n]):
            if device != "cpu": torch.cuda.synchronize()
            t0 = time.perf_counter()
            m([to_tensor(Image.open(p).convert("RGB")).to(device)])
            if device != "cpu": torch.cuda.synchronize()
            if i >= 10: times.append((time.perf_counter() - t0) * 1000)
    return round(float(np.mean(times)), 1)

for name, r in summary.items():
    if "speed_ms" in r:
        continue
    w = r["weights"]
    if r["family"] == "torchvision":
        r["speed_ms"] = {"gpu": bench_tv(name, w, "cuda"), "cpu": bench_tv(name, w, "cpu", n=25)}
    else:
        r["speed_ms"] = {"gpu": bench_ultralytics(w, 0), "cpu": bench_ultralytics(w, "cpu", n=25)}
    save_summary()
    print(name, r["speed_ms"])

## 8-1. 예측 예시
test 이미지 6장에 모델마다 예측 상자(conf ≥ 0.25, 초록)와 정답 상자(빨강)를 그려 `runs/<모델>/test_samples.jpg` 로 저장한다. 점수만으로 안 보이는 실수(끌개·사람을 잡음, 상자가 큼)를 눈으로 확인한다.

In [ ]:
from PIL import ImageDraw
TEST_IMAGES = sorted((DATA / "test" / "images").glob("*"))[:6]

def gt_boxes(p):
    img = Image.open(p); w, h = img.size
    rows = [r.split() for r in (DATA / "test" / "labels" / (p.stem + ".txt")).read_text().splitlines() if r.strip()]
    return [((float(cx) - float(bw) / 2) * w, (float(cy) - float(bh) / 2) * h,
             (float(cx) + float(bw) / 2) * w, (float(cy) + float(bh) / 2) * h) for _, cx, cy, bw, bh in rows]

def predict_boxes(name, r, p):
    if r["family"] == "torchvision":
        out = tv_cache[name]([to_tensor(Image.open(p).convert("RGB")).to(DEV)])[0]
        return [(b.tolist(), float(s)) for b, s in zip(out["boxes"].cpu(), out["scores"].cpu()) if s >= 0.25]
    res = ul_cache[name].predict(str(p), imgsz=IMGSZ, device=0, conf=0.25, verbose=False)[0]
    return [(b.tolist(), float(s)) for b, s in zip(res.boxes.xyxy.cpu(), res.boxes.conf.cpu())]

tv_cache, ul_cache = {}, {}
for name, r in summary.items():
    out_path = RESULTS / "runs" / name / "test_samples.jpg"
    if out_path.exists():
        continue
    if r["family"] == "torchvision":
        m, _ = build_tv(name); m.load_state_dict(torch.load(r["weights"], map_location="cpu")); tv_cache[name] = m.eval().to(DEV)
    else:
        ul_cache[name] = (RTDETR if "rtdetr" in name else YOLO)(r["weights"])
    tiles = []
    with torch.no_grad():
        for p in TEST_IMAGES:
            img = Image.open(p).convert("RGB"); d = ImageDraw.Draw(img)
            for b in gt_boxes(p):
                d.rectangle(b, outline=(255, 0, 0), width=2)
            for b, s in predict_boxes(name, r, p):
                d.rectangle(b, outline=(0, 255, 0), width=3); d.text((b[0] + 3, b[1] + 3), f"{s:.2f}", fill=(0, 255, 0))
            tiles.append(img.resize((320, 320)))
    sheet = Image.new("RGB", (320 * 3, 320 * 2))
    for i, t in enumerate(tiles):
        sheet.paste(t, ((i % 3) * 320, (i // 3) * 320))
    sheet.save(out_path, quality=90)
    print("saved", out_path)
    display(sheet)

## 8-2. 오탐·미탐, 실패 사례, 신뢰구간, 조건별 성능
- 이미지마다 conf ≥ 0.25 예측을 정답과 IoU 0.5 로 맞춰 **TP·FP(오탐)·FN(미탐)** 을 센다. 배경(정답 없는) 이미지의 오탐은 따로 센다.
- **실패 사례**: FP 나 FN 이 있는 이미지 목록을 `runs/<모델>/failures_<split>.csv` 로 저장한다.
- **신뢰구간**: 평가 이미지를 1000번 복원 추출(bootstrap)해 mAP50-95 의 95% 구간을 구한다. 두 모델 차이의 구간이 0 을 포함하면 **구분 불가**로 쓴다(평가셋이 작을 때 소수점 차이로 우열을 말하지 않기 위해).
- **조건별**: Drive `survivor_model_compare/eval_conditions.csv`(열: `split,image,condition`, condition 예: 저조도·가려짐·원거리·배경)가 있으면 조건마다 따로 센다. 없으면 건너뛴다.
- 이 칸의 mAP 는 `predict`(conf 0.001) 결과로 다시 계산하므로 6번 칸의 Ultralytics `val` 값과 소수 셋째 자리에서 다를 수 있다. 같은 칸 안에서는 모든 모델을 같은 방식으로 잰다.

In [ ]:
import csv
N_BOOT = 1000
rng = np.random.default_rng(SEED)

def split_gt(split, p):
    img = Image.open(p); w, h = img.size
    rows = [r.split() for r in (DATA / split / "labels" / (p.stem + ".txt")).read_text().splitlines() if r.strip()]
    return torch.tensor([[(float(cx) - float(bw) / 2) * w, (float(cy) - float(bh) / 2) * h,
                          (float(cx) + float(bw) / 2) * w, (float(cy) + float(bh) / 2) * h]
                         for _, cx, cy, bw, bh in rows], dtype=torch.float32).reshape(-1, 4)

def raw_predict(name, r, p):
    """conf 0.001, 최대 300개: (상자 xyxy 텐서, 점수 텐서)"""
    if r["family"] == "torchvision":
        out = tv_cache[name]([to_tensor(Image.open(p).convert("RGB")).to(DEV)])[0]
        return out["boxes"].cpu(), out["scores"].cpu()
    res = ul_cache[name].predict(str(p), imgsz=IMGSZ, device=0, conf=0.001, max_det=300, verbose=False)[0]
    return res.boxes.xyxy.cpu(), res.boxes.conf.cpu()

def match(gt, pb):
    correct = np.zeros((len(pb), len(IOUV)), dtype=bool)
    if len(pb) and len(gt):
        iou = box_iou(gt, pb).numpy()
        for k, th in enumerate(IOUV.tolist()):
            mt = np.array(np.nonzero(iou >= th)).T
            if mt.shape[0]:
                if mt.shape[0] > 1:
                    mt = mt[iou[mt[:, 0], mt[:, 1]].argsort()[::-1]]
                    mt = mt[np.unique(mt[:, 1], return_index=True)[1]]
                    mt = mt[np.unique(mt[:, 0], return_index=True)[1]]
                correct[mt[:, 1].astype(int), k] = True
    return correct

def map_of(items):
    if not any(it["n_gt"] for it in items):
        return float("nan")
    tp = np.concatenate([it["correct"] for it in items]); conf = np.concatenate([it["conf"] for it in items])
    tcls = np.zeros(sum(it["n_gt"] for it in items))
    _, _, p, r, _, ap, *_ = ap_per_class(tp, conf, np.zeros(len(conf)), tcls)
    return float(ap[0].mean())

conditions = {}
cond_csv = DRIVE_DIR / "eval_conditions.csv"
if cond_csv.exists():
    with open(cond_csv) as f:
        for row in csv.DictReader(f):
            conditions[(row["split"], Path(row["image"]).stem)] = row["condition"]
print("조건 표:", f"{len(conditions)}장" if conditions else "없음(조건별 분석 건너뜀)")

analysis = {}
with torch.no_grad():
    for name, r in summary.items():
        if name not in tv_cache and name not in ul_cache:
            if r["family"] == "torchvision":
                m, _ = build_tv(name); m.load_state_dict(torch.load(r["weights"], map_location="cpu")); tv_cache[name] = m.eval().to(DEV)
            else:
                ul_cache[name] = (RTDETR if "rtdetr" in name else YOLO)(r["weights"])
        analysis[name] = {}
        for split in ("valid", "test"):
            items, fail_rows = [], []
            for p in sorted((DATA / split / "images").glob("*")):
                gt = split_gt(split, p); pb, ps = raw_predict(name, r, p)
                keep = ps >= 0.25
                c50 = match(gt, pb[keep])[:, 0] if keep.any() else np.zeros(0, dtype=bool)
                tp_, fp_ = int(c50.sum()), int((~c50).sum()); fn_ = len(gt) - tp_
                items.append({"image": p.name, "n_gt": len(gt), "correct": match(gt, pb), "conf": ps.numpy(),
                              "tp": tp_, "fp": fp_, "fn": fn_, "cond": conditions.get((split, p.stem))})
                if fp_ or fn_:
                    fail_rows.append({"image": p.name, "gt": len(gt), "tp": tp_, "fp": fp_, "fn": fn_,
                                      "condition": conditions.get((split, p.stem), "")})
            boots = [map_of([items[i] for i in rng.integers(0, len(items), len(items))]) for _ in range(N_BOOT)]
            res = {"images": len(items), "instances": sum(it["n_gt"] for it in items),
                   "background_images": sum(1 for it in items if it["n_gt"] == 0),
                   "mAP50_95": round(map_of(items), 4),
                   "ci95": [round(float(np.nanpercentile(boots, 2.5)), 4), round(float(np.nanpercentile(boots, 97.5)), 4)],
                   "tp": sum(it["tp"] for it in items), "fp": sum(it["fp"] for it in items), "fn": sum(it["fn"] for it in items),
                   "fp_on_background": sum(it["fp"] for it in items if it["n_gt"] == 0),
                   "failure_images": [row["image"] for row in fail_rows]}
            if conditions:
                res["by_condition"] = {}
                for cond in sorted({it["cond"] for it in items if it["cond"]}):
                    sub = [it for it in items if it["cond"] == cond]
                    res["by_condition"][cond] = {"images": len(sub), "instances": sum(it["n_gt"] for it in sub),
                                                 "mAP50_95": round(map_of(sub), 4) if any(it["n_gt"] for it in sub) else None,
                                                 "tp": sum(it["tp"] for it in sub), "fp": sum(it["fp"] for it in sub),
                                                 "fn": sum(it["fn"] for it in sub)}
            res["_items"] = items
            analysis[name][split] = res
            with open(RESULTS / "runs" / name / f"failures_{split}.csv", "w", newline="") as f:
                w = csv.DictWriter(f, fieldnames=["image", "gt", "tp", "fp", "fn", "condition"]); w.writeheader(); w.writerows(fail_rows)
            print(f"{name} {split}: 인스턴스 {res['instances']}, 배경 {res['background_images']}장, mAP50-95 {res['mAP50_95']} "
                  f"(95% {res['ci95'][0]}~{res['ci95'][1]}), TP {res['tp']} FP {res['fp']} FN {res['fn']}, "
                  f"배경 오탐 {res['fp_on_background']}, 실패 이미지 {len(res['failure_images'])}장")

# 두 모델 차이: 같은 이미지 표본으로 동시에 복원 추출
def diff_ci(a, b, split):
    A, B = analysis[a][split]["_items"], analysis[b][split]["_items"]
    d = []
    for _ in range(N_BOOT):
        idx = rng.integers(0, len(A), len(A))
        d.append(map_of([A[i] for i in idx]) - map_of([B[i] for i in idx]))
    lo, hi = np.nanpercentile(d, 2.5), np.nanpercentile(d, 97.5)
    verdict = "구분 불가" if lo <= 0 <= hi else (f"{a} 우세" if lo > 0 else f"{b} 우세")
    return {"diff": round(analysis[a][split]["mAP50_95"] - analysis[b][split]["mAP50_95"], 4),
            "ci95": [round(float(lo), 4), round(float(hi), 4)], "verdict": verdict}

names = list(analysis)
pairwise = {f"{a} vs {b}": {s: diff_ci(a, b, s) for s in ("valid", "test")}
            for i, a in enumerate(names) for b in names[i + 1:]}
for k, v in pairwise.items():
    print(k, {s: f"{x['diff']:+} (95% {x['ci95'][0]}~{x['ci95'][1]}) {x['verdict']}" for s, x in v.items()})
(RESULTS / "analysis.json").write_text(json.dumps(
    {"per_model": {n: {s: {k: v for k, v in r.items() if k != "_items"} for s, r in a.items()} for n, a in analysis.items()},
     "pairwise": pairwise}, ensure_ascii=False, indent=2))

## 9. 결과 표와 그래프
Colab 에는 한글 글꼴이 없어 그래프 글자는 영어로 둔다.

In [ ]:
import matplotlib.pyplot as plt
gpu_name = torch.cuda.get_device_name(0)
rows = []
for name, r in summary.items():
    rows.append({"모델": name, "계열": r["family"], "파라미터(M)": r["params_M"], "입력": r["hp"]["imgsz"],
                 "batch": r["hp"]["batch"], "학습(분)": r["train_minutes"], "best epoch": r["best_epoch"],
                 "val 인스턴스": info["splits"]["valid"]["boxes"], "test 인스턴스": info["splits"]["test"]["boxes"],
                 "val P": r["val"]["precision"], "val R": r["val"]["recall"], "val mAP50": r["val"]["mAP50"],
                 "val mAP50-95": r["val"]["mAP50_95"], "test mAP50": r["test"]["mAP50"],
                 "test mAP50-95": r["test"]["mAP50_95"],
                 "GPU ms/장": r.get("speed_ms", {}).get("gpu"), "CPU ms/장": r.get("speed_ms", {}).get("cpu")})
df = pd.DataFrame(rows).sort_values("val mAP50-95", ascending=False).reset_index(drop=True)
df.to_csv(RESULTS / "results.csv", index=False)
display(df)

fig, ax = plt.subplots(1, 2, figsize=(14, 5))
x = np.arange(len(df))
ax[0].bar(x - 0.2, df["val mAP50-95"], 0.4, label="val"); ax[0].bar(x + 0.2, df["test mAP50-95"], 0.4, label="test")
ax[0].set_xticks(x, df["모델"], rotation=45, ha="right"); ax[0].set_ylabel("mAP50-95"); ax[0].legend()
ax[0].set_ylim(max(0, df[["val mAP50-95", "test mAP50-95"]].min().min() - 0.1), 1); ax[0].set_title("Accuracy (mAP50-95)")
for _, r in df.iterrows():
    ax[1].scatter(r["CPU ms/장"], r["val mAP50-95"]); ax[1].annotate(r["모델"], (r["CPU ms/장"], r["val mAP50-95"]))
ax[1].set_xlabel("CPU ms/image (Colab CPU, end-to-end)"); ax[1].set_ylabel("val mAP50-95"); ax[1].set_title("Accuracy vs CPU latency")
plt.tight_layout(); plt.savefig(RESULTS / "compare.png", dpi=150); plt.show()

plt.figure(figsize=(10, 5))
for name, r in summary.items():
    d = RESULTS / "runs" / name
    if r["family"] == "torchvision":
        h = pd.read_csv(d / "history.csv"); plt.plot(h["epoch"], h["mAP50_95"], label=name, ls="--")
    else:
        h = pd.read_csv(d / "results.csv"); h.columns = [c.strip() for c in h.columns]
        plt.plot(h["epoch"], h["metrics/mAP50-95(B)"], label=name)
plt.xlabel("epoch"); plt.ylabel("val mAP50-95"); plt.title("Training curves"); plt.legend(); plt.grid(alpha=.3)
plt.savefig(RESULTS / "curves.png", dpi=150); plt.show()

## 10. Notion 정리용 요약
아래 출력을 그대로 Notion 에 붙여 넣으면 표가 된다. `notion_summary.md` 로도 Drive 에 저장된다.

In [ ]:
hp_ul = {k: v for k, v in HP.items() if k not in ("verbose", "plots", "workers", "device")}
aug_df = pd.DataFrame([{"모델": n, **r.get("online_aug", {})} for n, r in summary.items()]).fillna("-")
hp_df = pd.DataFrame([{"모델": n, "optimizer(실제)": r.get("optimizer_used"), "lr0": r.get("lr0_used"),
                       "lrf": r.get("lrf"), "momentum": r.get("momentum_used"), "weight_decay": r.get("weight_decay"),
                       "warmup_epochs": r.get("warmup_epochs"), "epochs": r["hp"]["epochs"], "batch": r["hp"]["batch"],
                       "입력": r["hp"]["imgsz"], "best epoch": r["best_epoch"], "학습(분)": r["train_minutes"]}
                      for n, r in summary.items()])
aug_df.to_csv(RESULTS / "augmentation.csv", index=False); hp_df.to_csv(RESULTS / "hyperparameters.csv", index=False)
lines = [
    "# survivor 탐지 모델 비교 (v2 데이터셋)", "",
    f"- 실행: Colab {gpu_name}, ultralytics {ultralytics.__version__}, torchvision {torchvision.__version__}, torch {torch.__version__}",
    f"- 데이터셋: `{info['zip']}` (sha256 `{info['zip_sha256'][:12]}…`)",
    "- 분할: " + ", ".join(f"{s} {v['images']}장(상자 {v['boxes']}, 빈 이미지 {v['empty_images']})" for s, v in info["splits"].items()),
    "- Roboflow 전처리·증강(train 에만, 3배): " + "; ".join(info["roboflow_preprocessing_and_augmentation"]),
    "- 공통 하이퍼파라미터(Ultralytics): " + ", ".join(f"{k}={v}" for k, v in hp_ul.items()) + " (RT-DETR-l 은 batch 8)",
    *(["- torchvision: SGD, cosine, warmup 1 epoch, 좌우 뒤집기만. Faster R-CNN lr 0.01 wd 1e-4 입력 640 / SSDLite lr 0.01 wd 4e-5 입력 320"] if TORCHVISION_MODELS else []),
    "- 평가: IoU 0.5:0.95, conf 0.001, 최대 300개, Ultralytics ap_per_class 로 모든 모델 동일. 속도는 배치 1 끝에서 끝까지",
    "- valid·test 는 학습과 따로 찍은 영상(VAL·TEST)에서만 뽑음. 둘은 같은 시간대·같은 사람이라 조건이 비슷함",
    "", "## 결과", "", df.to_markdown(index=False), "",
    "## 오탐·미탐과 신뢰구간 (conf ≥ 0.25, IoU 0.5)", "",
    "| 모델 | split | 이미지 | 인스턴스 | 배경 | mAP50-95 (95% 구간) | TP | FP | FN | 배경 오탐 | 실패 이미지 |",
    "| --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- |",
    *[f"| {n} | {s} | {r['images']} | {r['instances']} | {r['background_images']} | {r['mAP50_95']} ({r['ci95'][0]}~{r['ci95'][1]}) "
      f"| {r['tp']} | {r['fp']} | {r['fn']} | {r['fp_on_background']} | {len(r['failure_images'])} |"
      for n, a in analysis.items() for s, r in a.items()], "",
    "**모델 간 차이** (같은 이미지로 bootstrap, 95% 구간이 0 을 포함하면 구분 불가)", "",
    *[f"- {k}: " + ", ".join(f"{s} {x['diff']:+} ({x['ci95'][0]}~{x['ci95'][1]}) **{x['verdict']}**" for s, x in v.items())
      for k, v in pairwise.items()], "",
    *(["**조건별**", "", "| 모델 | split | 조건 | 이미지 | 인스턴스 | mAP50-95 | TP | FP | FN |", "| --- | --- | --- | --- | --- | --- | --- | --- | --- |",
       *[f"| {n} | {s} | {c} | {x['images']} | {x['instances']} | {x['mAP50_95']} | {x['tp']} | {x['fp']} | {x['fn']} |"
         for n, a in analysis.items() for s, r in a.items() for c, x in r.get("by_condition", {}).items()], ""]
      if conditions else ["조건별 분석: eval_conditions.csv 없음(미실행)", ""]),
    "실패 이미지 목록은 `runs/<모델>/failures_valid.csv`, `failures_test.csv`.", "",
    "## 데이터 증강", "",
    "**1) Roboflow 오프라인 증강** (train 이미지를 3배로 만들어 저장, valid/test 에는 없음)", "",
    *[f"- {a}" for a in info["roboflow_preprocessing_and_augmentation"]], "",
    "**2) 학습 중 온라인 증강** (매 배치 무작위로 적용, 모델별 실제 값)", "",
    aug_df.to_markdown(index=False), "",
    *(["torchvision 모델은 좌우 뒤집기(0.5)만 쓴다. YOLO 계열의 mosaic·HSV·scale 등이 없으므로 비교할 때 감안한다.", ""]
      if TORCHVISION_MODELS else []),
    "## 모델별 하이퍼파라미터", "", hp_df.to_markdown(index=False), "",
    "Ultralytics 모델의 나머지 값 전체는 `results/runs/<모델>/args.yaml` 과 `summary.json` 의 `train_args` 에 있다.", "",
    "## 그래프 (Drive `survivor_model_compare/results/`)", "",
    "- `compare.png`: 모델별 mAP50-95(val/test), 정확도 vs CPU 속도",
    "- `curves.png`: 모델별 val mAP50-95 학습 곡선",
    "- `runs/<모델>/results.png`: 모델별 손실·지표 곡선 (Ultralytics 는 PR·F1 곡선, 혼동 행렬도 같은 폴더)",
    "- `runs/<모델>/test_samples.jpg`: test 6장 예측 예시(초록 예측, 빨강 정답)", "",
    *(["## 실패한 모델", "", *[f"- {n}: {e}" for n, e in failures.items()], ""] if failures else []),
    "## 모델별 가중치 해시", "",
    "| 모델 | 사전학습 | best.pt sha256 |", "| --- | --- | --- |",
]
for name, r in summary.items():
    pre = r.get("pretrained_sha256", r.get("pretrained", ""))
    lines.append(f"| {name} | `{str(pre)[:16]}…` | `{r['best_sha256'][:16]}…` |")
text = "\n".join(lines)
(RESULTS / "notion_summary.md").write_text(text)
print(text)

## 다음 단계
- 학습에 쓰지 않은 **별도 검증 영상**(다른 날·다른 사람·다른 조명)으로 상위 모델을 다시 평가한다.
- 상위 모델은 PC3 에서 실제 fps 를 잰다(여기 CPU 값은 Colab CPU 기준).
- 배포할 모델은 저장소 `assets/manifest/` 에 출처·데이터셋 버전·가중치 해시·`class_labels: ["survivor"]` 를 기록한다.